# Global House Price Analysis
How does floor area relate to listing price after accounting for bedrooms, bathrooms and city?

This public demonstration adapts collaborative ST236 coursework. All listings and currency conversion factors below are illustrative.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(42)
cities = {
    "London": (1.0, 1.0, 8000),
    "Lisbon": (0.85, 1.0, 4300),
    "Mumbai": (0.01, 0.09290304, 3100),
    "Perth": (0.52, 1.0, 4200),
}
tables = []
for city, (gbp_per_unit, sqm_per_unit, price_scale) in cities.items():
    area = rng.uniform(35, 190, 240)
    bedrooms = np.clip((area / 38 + rng.normal(0, 0.55, len(area))).round(), 1, 5)
    bathrooms = np.clip((area / 65 + rng.normal(0, 0.5, len(area))).round(), 1, 4)
    price = price_scale * 90 * (area / 90) ** 0.75
    price *= np.exp(0.06 * (bedrooms - 2) + 0.12 * (bathrooms - 2)
                    + rng.normal(0, 0.23, len(area)))
    table = pd.DataFrame({
        "city": city, "area_local": area / sqm_per_unit,
        "price_local": price / gbp_per_unit, "bedrooms": bedrooms,
        "bathrooms": bathrooms, "gbp_per_unit": gbp_per_unit,
        "sqm_per_unit": sqm_per_unit,
    })
    table.loc[0, "area_local"] = np.nan
    tables.append(pd.concat([table, table.iloc[[1]]], ignore_index=True))


## Standardise units and clean listings

Convert prices to GBP and areas to square metres. Remove the injected duplicate listings and rows without the fields required by the regression.

In [ ]:
raw = pd.concat(tables, ignore_index=True)
listings = raw.drop_duplicates().dropna().assign(
    area_sqm=lambda rows: rows.area_local * rows.sqm_per_unit,
    price_gbp=lambda rows: rows.price_local * rows.gbp_per_unit,
)
listings = listings.query("area_sqm > 0 and price_gbp > 0").assign(
    log_area=lambda rows: np.log(rows.area_sqm),
    log_price=lambda rows: np.log(rows.price_gbp),
)
print(f"{len(raw)} raw rows; {len(listings)} usable listings")


## Fit and evaluate the regression

Compare a city-only baseline with a log-linear OLS model on the same holdout. The expanded model uses a common area slope and a separate intercept for each city. A random split stratified by city measures performance on similar synthetic listings.

In [ ]:
train, test = train_test_split(
    listings, test_size=0.25, random_state=42, stratify=listings.city
)
city_columns = ["Lisbon", "London", "Mumbai", "Perth"]

def design(rows, expanded=True):
    city = pd.Categorical(rows.city, categories=city_columns)
    indicators = pd.get_dummies(city, drop_first=True, dtype=float).to_numpy()
    if expanded:
        return np.column_stack((rows[["log_area", "bedrooms", "bathrooms"]], indicators))
    return indicators

baseline = LinearRegression().fit(design(train, expanded=False), train.log_price)
model = LinearRegression().fit(design(train), train.log_price)
for name, predicted in [
    ("City only", baseline.predict(design(test, expanded=False))),
    ("Expanded OLS", model.predict(design(test))),
]:
    residual = test.log_price - predicted
    print(f"{name}: holdout log-price RMSE {np.sqrt(np.mean(residual ** 2)):.3f}")
print(f"Estimated floor-area elasticity: {model.coef_[0]:.3f}")


## Visualise the fitted relationship

The curves hold bedrooms and bathrooms at two. They plot exponentiated fitted log prices, which represent conditional geometric means. Differences between cities in this demonstration reflect the synthetic data-generating process.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.4), layout="constrained")
area = np.linspace(45, 170, 80)
for city, group in listings.groupby("city"):
    scenario = pd.DataFrame({
        "city": city, "log_area": np.log(area), "bedrooms": 2, "bathrooms": 2,
    })
    line, = ax.plot(area, np.exp(model.predict(design(scenario))), label=city)
    ax.scatter(group.area_sqm, group.price_gbp, color=line.get_color(), s=8, alpha=0.12)
ax.set(title="Floor area and listing price | synthetic data",
       xlabel="Floor area (square metres)", ylabel="Illustrative listing price (GBP)")
ax.legend()
ax.grid(alpha=0.2)
fig.savefig(Path("housing-demo.png"), dpi=100)
plt.show()
